In [1]:
import pandas as pd
import torch
from torch import nn

SIDE_LIMIT = 1.5
NUM_EPOCHS = 2000

torch.manual_seed(42)

In [2]:
closing_prices = pd.read_csv("rosneft_close.csv", sep=";")["CLOSE"].tolist()

print("Количество цен:", len(closing_prices))
print("Первые десять цен:", closing_prices[:10])

Количество цен: 675
Первые десять цен: [527.4, 508.0, 516.9, 510.45, 517.25, 524.15, 531.85, 530.55, 532.25, 532.45]


In [3]:
def linear_slope(values):
    time_steps = [1, 2, 3]
    avg_time = sum(time_steps) / 3
    avg_value = sum(values) / 3

    cross_sum = sum(
        (time_step - avg_time) * (value - avg_value)
        for time_step, value in zip(time_steps, values)
    )
    time_sum = sum((time_step - avg_time) ** 2 for time_step in time_steps)
    return cross_sum / time_sum

def trend_label(forecast_prices):
    coefficient = linear_slope(forecast_prices)
    if coefficient > SIDE_LIMIT:
        return 1
    elif coefficient < -SIDE_LIMIT:
        return -1
    return 0

In [4]:
def prepare_samples(values):
    features = []
    labels = []

    for offset in range(len(values) - 12):
        past_prices = values[offset:offset + 10]
        next_prices = values[offset + 10:offset + 13]

        features.append([value - past_prices[-1] for value in past_prices])

        labels.append(trend_label(next_prices) + 1)

    return (
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long),
    )

In [5]:
split_index = int(len(closing_prices) * 0.8)
X_train, y_train = prepare_samples(closing_prices[:split_index])
X_test, y_test = prepare_samples(closing_prices[split_index:])

print("Обучающие входы:", tuple(X_train.shape))
print("Обучающие ответы:", tuple(y_train.shape))
print("Тестовые входы:", tuple(X_test.shape))
print("Тестовые ответы:", tuple(y_test.shape))

Обучающие входы: (528, 10)
Обучающие ответы: (528,)
Тестовые входы: (123, 10)
Тестовые ответы: (123,)


In [6]:
classifiers = {
    "Сеть 1: 10 → 24 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 3),
    ),
    "Сеть 2: 10 → 32 → 3, Sigmoid": nn.Sequential(
        nn.Linear(10, 32),
        nn.Sigmoid(),
        nn.Linear(32, 3),
    ),
    "Сеть 3: 10 → 24 → 12 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 12),
        nn.Tanh(),
        nn.Linear(12, 3),
    ),
}

In [7]:
final_errors = {}

for title, classifier in classifiers.items():
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)

    classifier.train()
    for iteration in range(NUM_EPOCHS):
        optimizer.zero_grad()
        logits = classifier(X_train)
        error = criterion(logits, y_train)
        error.backward()
        optimizer.step()

    final_errors[title] = error.item()

In [8]:
def compute_accuracy(classifier, features, labels):
    classifier.eval()
    with torch.no_grad():
        predicted_labels = classifier(features).argmax(dim=1)
    return (predicted_labels == labels).float().mean().item()

evaluation = []
for title, classifier in classifiers.items():
    evaluation.append({
        "Сеть": title,
        "Ошибка обучения": final_errors[title],
        "Точность на обучении, %": compute_accuracy(classifier, X_train, y_train) * 100,
        "Точность на тесте, %": compute_accuracy(classifier, X_test, y_test) * 100,
    })

pd.DataFrame(evaluation).round(2)

,Сеть,Ошибка обучения,"Точность на обучении, %","Точность на тесте, %"
0,"Сеть 1: 10 → 24 → 3, Tanh",0.47,79.92,29.27
1,"Сеть 2: 10 → 32 → 3, Sigmoid",0.30,89.96,35.77
2,"Сеть 3: 10 → 24 → 12 → 3, Tanh",0.16,93.37,25.20
